In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
        )

        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        shortcut_layer = self.shortcut(x)
        out = self.conv1(x)
        out = self.conv2(out) + shortcut_layer
        out = self.relu(out)
        return out

In [ ]:
class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=4, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )

        self.layer1 = ResidualBlock(16, 32)
        self.layer2 = ResidualBlock(32, 64)
        self.layer3 = ResidualBlock(64, 128)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fully_connected = nn.Linear(128, num_classes)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)

        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fully_connected(out)
        return out

In [ ]:
max_val_acc = -1

In [ ]:
import sys

sys.path.insert(0, "../scripts")  # Add parent folder to Python path
from preprocess import preprocess
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

x_tensor, y_tensor, mapping = preprocess()

# train test split
x_train, x_, y_train, y_ = train_test_split(
    x_tensor,
    y_tensor,
    test_size=0.15,
    random_state=90,
    stratify=y_tensor,
)
x_val, x_test, y_val, y_test = train_test_split(
    x_,
    y_,
    test_size=0.33,
    random_state=90,
    stratify=y_,
)

# put it to train val test loader
traindataset = TensorDataset(x_train, y_train)
trainloader = DataLoader(traindataset, batch_size=4, shuffle=True)

valdataset = TensorDataset(x_val, y_val)
valloader = DataLoader(valdataset, batch_size=4, shuffle=False)

testdataset = TensorDataset(x_test, y_test)
testloader = DataLoader(testdataset, batch_size=4, shuffle=False)


model = ResNet()

# training
model = model.to("cuda")
validation_accuracies = []
training_loss = []

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), 0.0001)
device = "cuda"
print("Starting Training...")

epo = 20
for epoch in range(epo):
    model.train()
    running_loss = 0.0

    for x, y in trainloader:
        x, y = x.to("cuda"), y.to("cuda")

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(trainloader)
    training_loss.append(avg_train_loss)

    print(f"Epoch [{epoch+1}/{epo}], Loss: {avg_train_loss:.4f}")

    # Evaluate model on validation data
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in valloader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    val_accuracy = 100 * correct / total
    max_val_acc = max(val_accuracy, max_val_acc)

    if val_accuracy == max_val_acc:
        print("Saving at accuracy", val_accuracy)
        torch.save(model.state_dict(), "resnetmodel.pth")

    print(f"Val Accuracy: {val_accuracy:.2f}%\n")
    validation_accuracies.append(val_accuracy)

In [ ]:
loaded_model = ResNet()
loaded_model.load_state_dict(torch.load("resnetmodel.pth"))

In [ ]:
# final evaluation on test data
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for x, y in testloader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        _, predicted = torch.max(outputs.data, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

test_accuracy = 100 * correct / total
print("Test Acc", test_accuracy)
